# Managing Media Files

This notebook demonstrates the MediaDir functionality from `mediatools`, which provides a powerful recursive data structure for organizing and managing media files in filesystem directories.

**What you'll learn:**
- How to scan directories for media files (videos, images, other files)
- Navigate and work with nested directory structures
- Access and process media files at individual and batch levels
- Compare directories and detect changes
- Integration patterns with other mediatools modules

**Prerequisites:**
- Basic Python knowledge
- Familiarity with pathlib for file system operations
- Some media files to experiment with (we'll create mock examples)

**Key Concepts:**
- **MediaDir**: Recursive directory structure containing videos, images, and subdirectories
- **File Collections**: VideoFiles, ImageFiles, and other file containers
- **Metadata Integration**: Each file type provides rich metadata access
- **Path Flexibility**: Support for both absolute and relative path handling

#### Setup and Mock Data Creation

Since this is a demonstration notebook, we'll create a realistic directory structure with mock media files. This allows you to run the examples without needing your own media collection.

In [1]:
from pathlib import Path

# get environment variables for testing
import pydantic_settings
class Settings(pydantic_settings.BaseSettings):
    test_zip_file_url: str
settings = Settings()

# download zip file and dump it into a temporary directory
from example_helpers import RemoteZipDownloader
zd = RemoteZipDownloader.from_url(settings.test_zip_file_url)
temp_path = zd.path

# show downloaded files
print(f'Downloaded files in {temp_path}')
for item in sorted(temp_path.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(temp_path)}")
temp_path

Downloaded files in /tmp/tmprx75laht
  battles/totk_lynels/how_to_battle.m4a
  battles/totk_lynels/how_to_battle.mp4
  battles/totk_lynels/how_to_battle.txt
  battles/totk_lynels/how_to_battle_thumb.jpg
  totk_builds/op_builds.m4a
  totk_builds/op_builds.mp4
  totk_builds/op_builds.txt
  totk_builds/op_builds_thumb.jpg


PosixPath('/tmp/tmprx75laht')

In [2]:
import mediatools

## Scanning for Media Files

`MediaDir` instances are the primary way to track a wide range of media files on disk.

The primary way to create a MediaDir is by scanning a filesystem directory. MediaDir will automatically categorize files into videos, images, and other files based on their extensions.

In [3]:
media_dir = mediatools.scan_directory(temp_path)
print(media_dir.display())

/tmp/tmprx75laht/
├── battles/
│       ├── [V] how_to_battle.mp4
│       ├── [I] how_to_battle_thumb.jpg
│       ├── [F] how_to_battle.m4a
│       └── [F] how_to_battle.txt
└── totk_builds/
    ├── [V] op_builds.mp4
    ├── [I] op_builds_thumb.jpg
    ├── [F] op_builds.m4a
    └── [F] op_builds.txt


The `MediaDir.from_path` constructor is equivalent.

In [4]:
mediatools.MediaDir.from_path(temp_path)

MediaDir("/tmp/tmprx75laht")

Either method accepts `video_ext` and `image_ext` parameters to control which file types are considered as video or images. From the `display` output see that the .mp4 files are no longer seen as video files (they show `F` instead of `V`).

In [5]:
md = mediatools.scan_directory(
    temp_path,
    video_ext=[],  # Only MP4 and MOV files
    image_ext=['.jpg', '.jpeg']  # Only JPEG files
)
print(md.display())

/tmp/tmprx75laht/
├── battles/
│       ├── [I] how_to_battle_thumb.jpg
│       ├── [F] how_to_battle.m4a
│       ├── [F] how_to_battle.mp4
│       └── [F] how_to_battle.txt
└── totk_builds/
    ├── [I] op_builds_thumb.jpg
    ├── [F] op_builds.m4a
    ├── [F] op_builds.mp4
    └── [F] op_builds.txt


The `ignore_path` allows you to ignore particular paths. Note that this is applied recursively, so any matching path will be ignored as well as all of its subdirectories.

Here I ignore all directories that start with "b".

In [6]:
md = mediatools.scan_directory(
    temp_path,
    ignore_path = lambda p: p.name == 'battles',
)
print(md.display())

/tmp/tmprx75laht/
└── totk_builds/
    ├── [V] op_builds.mp4
    ├── [I] op_builds_thumb.jpg
    ├── [F] op_builds.m4a
    └── [F] op_builds.txt


The `path` attribute allows you to access the path of a given `MediaDir`.

In [7]:
media_dir.path

PosixPath('/tmp/tmprx75laht')

## Navigating Directory Trees

Directories are trees, so you can navigate them as such.

Access subdirectories using the `subdirs` attribute. This is a dictionary of subdir names mapped to `MediaDir` instances.

In [8]:
print(f"{media_dir} has {len(media_dir.subdirs)} subdirectories:")
for subdir_name, subdir in media_dir.subdirs.items():
    print(f"  {subdir_name}: {subdir.path}")

MediaDir("/tmp/tmprx75laht") has 2 subdirectories:
  battles: /tmp/tmprx75laht/battles
  totk_builds: /tmp/tmprx75laht/totk_builds


Each `MediaDir` also has a parent.

In [9]:
for subdir in media_dir.subdirs.values():
    print(subdir.parent)

MediaDir("/tmp/tmprx75laht")
MediaDir("/tmp/tmprx75laht")


Note that the root instance parent is `None`.

In [10]:
print(media_dir.parent)

None


The fact that this is a recursive data structure means it is easy to write recursive functions.

In [11]:
def print_tree(mdir: mediatools.MediaDir, level: int = 0):
    print(f"{'  ' * level}- {mdir.path.name}")
    for subdir in mdir.subdirs.values():
        print_tree(subdir, level + 1)

print_tree(media_dir)

- tmprx75laht
  - battles
    - totk_lynels
  - totk_builds


In [12]:
def count_files(mdir: mediatools.MediaDir) -> int:
    count = len(mdir.videos) + len(mdir.images) + len(mdir.other_files)
    for subdir in mdir.subdirs.values():
        count += count_files(subdir)
    return count
count_files(media_dir)

8

### Navigate Directories as Paths

There are also more path-centric methods for interacting with the media directories.

You can use subscripts to access child directories. Note that you cannot use this to access files - only subdirectories.

In [13]:
media_dir["battles"]

MediaDir("/tmp/tmprx75laht/battles")

In [14]:
media_dir["battles"]['totk_lynels']

MediaDir("/tmp/tmprx75laht/battles/totk_lynels")

You can also use the `subdir` method to access children. Note that it accepts a variable number of arguments.

In [15]:
media_dir.subdir("battles", "totk_lynels")

MediaDir("/tmp/tmprx75laht/battles/totk_lynels")

The `subdir` method also accepts full relative paths.

In [16]:
media_dir.subdir(Path("battles") / "totk_lynels")

MediaDir("/tmp/tmprx75laht/battles/totk_lynels")

## Working with Media Files

`MediaDir` instances automatically track videos and images in `VideoFile` and `ImageFile` instances, and all other files are wrapped in `NonMediaFile` instances. You can access videos through `.videos`, images through `.images`, and other files through `.other_files`.

This is a set of methods and properties you can use to access files from a `MediaDir`:


**Properties for Current Directory**

- `videos` (`list[VideoFile]`): videos in the represented directory only.
- `images` (`list[ImageFile]`): images in the represented directory only.
- `other_files` (`list[NonMediaFile]`): all non-media files in the directory only.

**Path Methods for Current Directory**
- `video_paths()` (`list[Path]`): list of video file paths in the directory only.
- `image_paths()` (`list[Path]`): list of image file paths in the directory only.

In [17]:
ex_dir = media_dir["battles/totk_lynels"]
ex_dir.videos

[VideoFile(path=PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4'), meta={})]

In [18]:
ex_dir.images

[ImageFile(path=PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg'), meta={})]

In [19]:
ex_dir.image_paths()

[PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg')]

In [20]:
ex_dir.other_files

[NonMediaFile(path=PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.txt'), meta={}),
 NonMediaFile(path=PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.m4a'), meta={})]

In [21]:
ex_dir.video_paths()

[PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4')]

**Get Specific Files by Path**
- `get_video(path)` (`VideoFile`): retrieve a specific video file by its path.
- `get_image(path)` (`ImageFile`): retrieve a specific image file by its path.
- `get_nonmedia(path)` (`NonMediaFile`): retrieve a specific non-media file by its path.

In [22]:
media_dir['totk_builds'].other_files

[NonMediaFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.m4a'), meta={}),
 NonMediaFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.txt'), meta={})]

In [23]:
media_dir.get_video(temp_path / "totk_builds/op_builds.mp4")

VideoFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4'), meta={})

In [24]:
media_dir.get_image(temp_path / "totk_builds/op_builds_thumb.jpg")

ImageFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds_thumb.jpg'), meta={})

In [25]:
media_dir.get_nonmedia(temp_path / "totk_builds/op_builds.txt")

NonMediaFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.txt'), meta={})

You can also retrieve all video or image files recursively from a media directory.

**Recursive Methods (Include Subdirectories)**
- `all_video_files()` (`list[VideoFile]`): all video files recursively across directory tree.
- `all_image_files()` (`list[ImageFile]`): all image files recursively across directory tree.
- `all_video_paths()` (`list[Path]`): all video file paths recursively across directory tree.
- `all_image_paths()` (`list[Path]`): all image file paths recursively across directory tree.
- `all_file_paths()` (`list[Path]`): all file paths (including non-media) recursively across directory tree.
- `all_media_paths()` (`list[Path]`): all media file paths (videos + images) recursively across directory tree.


In [26]:
media_dir.all_image_paths()

[PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds_thumb.jpg')]

In [27]:
media_dir.all_file_paths()

[PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4'),
 PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds_thumb.jpg')]

In [28]:
media_dir.all_media_paths()

[PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4'),
 PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds_thumb.jpg')]

In [29]:
media_dir.all_video_files()

[VideoFile(path=PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4'), meta={}),
 VideoFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4'), meta={})]

In [30]:
media_dir.all_image_files()

[ImageFile(path=PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg'), meta={}),
 ImageFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds_thumb.jpg'), meta={})]

In [31]:
media_dir.all_video_paths()

[PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4')]

All file instance types have a `stat` method that returns a `FileStatResult` type, which is essentially a `pydantic.BaseType` containing the same information as `os.stat_result` with some convenient methods.

In [32]:
ex_vid = ex_dir.videos[0]
ex_vid.path

PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4')

In [33]:
stat = ex_vid.stat()
stat

FileStatResult(st_mode=33204, st_ino=44437806, st_dev=66309, st_nlink=1, st_uid=1000, st_gid=1000, st_size=1478840, st_atime=1777741602.4067485, st_mtime=1777741602.4107487, st_ctime=1777741602.4107487, st_atime_ns=1777741602406748554, st_mtime_ns=1777741602410748603, st_ctime_ns=1777741602410748603, st_blksize=4096, st_blocks=2896, st_rdev=0, st_birthtime=None)

In [34]:
stat.size_str(), stat.modified_at_str(), stat.accessed_at_str(), stat.changed_at_str()

('1.48 MB',
 '2026-05-02 17:06:42 UTC',
 '2026-05-02 17:06:42 UTC',
 '2026-05-02 17:06:42 UTC')

## Directory Comparison and Synchronization

### Detecting Changes Between Directory States

In [35]:
temp_path / "totk_builds/op_builds.mp4"

PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4')

In [36]:
media_dir['totk_builds'].videos

[VideoFile(path=PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4'), meta={})]

In [37]:
import shutil
import tempfile

def create_modified_directory():
    """Create a slightly modified version of our demo directory"""
    modified_root = Path(tempfile.mkdtemp(prefix="mediatools_modified_"))
    shutil.copytree(temp_path, modified_root, dirs_exist_ok=True)

    # make some changes
    shutil.move(modified_root / "totk_builds/op_builds.mp4", modified_root / "new_vid.mp4")
    (modified_root / "new_dir").mkdir()
    shutil.move(modified_root / "totk_builds/op_builds_thumb.jpg", modified_root / "new_dir/new_thumb.jpg")

    return modified_root

# Create modified directory and scan it
modified_root = create_modified_directory()
modified_media_dir = mediatools.scan_directory(modified_root)
print(modified_media_dir.display())

/tmp/mediatools_modified_icxpolo4/
├── [V] new_vid.mp4
├── battles/
│       ├── [V] how_to_battle.mp4
│       ├── [I] how_to_battle_thumb.jpg
│       ├── [F] how_to_battle.m4a
│       └── [F] how_to_battle.txt
├── new_dir/
│   └── [I] new_thumb.jpg
└── totk_builds/
    ├── [F] op_builds.m4a
    └── [F] op_builds.txt


In [38]:
# Compare file counts
len(media_dir.all_file_paths()), len(modified_media_dir.all_file_paths())

(4, 4)

In [39]:
# Compare the two directory structures
removed_files, added_files = media_dir.file_diff(modified_media_dir)

In [40]:
# Files removed in modified version
removed_files

{PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4'),
 PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds.mp4'),
 PosixPath('/tmp/tmprx75laht/totk_builds/op_builds_thumb.jpg')}

In [41]:
# Files added in modified version
added_files

{PosixPath('/tmp/mediatools_modified_icxpolo4/battles/totk_lynels/how_to_battle.mp4'),
 PosixPath('/tmp/mediatools_modified_icxpolo4/battles/totk_lynels/how_to_battle_thumb.jpg'),
 PosixPath('/tmp/mediatools_modified_icxpolo4/new_dir/new_thumb.jpg'),
 PosixPath('/tmp/mediatools_modified_icxpolo4/new_vid.mp4')}

In [42]:
# Find directories that have changes
changed_dirs = media_dir.get_changed_dirs(modified_media_dir)

In [43]:
# Show directories with changes
[changed_dir.path.relative_to(temp_path) for changed_dir in changed_dirs]

[PosixPath('totk_builds')]

## Serialization and Data Export

### Converting MediaDir to Dictionary Formats

In [44]:
# Convert to dictionary representation
media_dict = media_dir.to_dict()

In [45]:
# Show top-level dictionary structure
{k: (list(v.keys()) if k == 'subdirs' and isinstance(v, dict) 
     else len(v) if isinstance(v, list) 
     else v) for k, v in media_dict.items()}

{'path': '/tmp/tmprx75laht',
 'videos': 0,
 'images': 0,
 'other_files': 0,
 'subdirs': 2,
 'meta': {}}

## Practical Use Cases and Workflows

### 1. Media Library Organization

In [46]:
def analyze_media_library(media_dir):
    """Generate a comprehensive analysis of a media library"""
    
    analysis = {
        'total_directories': len(list(media_dir.path.rglob('*'))) if media_dir.path.exists() else 0,
        'total_videos': len(media_dir.all_video_files()),
        'total_images': len(media_dir.all_image_files()),
        'total_other_files': len(media_dir.all_file_paths()) - len(media_dir.all_media_paths()),
        'directory_breakdown': {}
    }
    
    # Analyze each subdirectory
    for subdir_name, subdir in media_dir.subdirs.items():
        analysis['directory_breakdown'][subdir_name] = {
            'videos': len(subdir.all_video_files()),
            'images': len(subdir.all_image_files()),
            'subdirectories': len(subdir.subdirs)
        }
    
    return analysis

In [47]:
# Analyze our demo library
analysis = analyze_media_library(media_dir)

In [48]:
# Show total counts
analysis['total_videos'], analysis['total_images'], analysis['total_other_files']

(2, 2, 0)

In [49]:
# Show directory breakdown
analysis['directory_breakdown']

{'battles': {'videos': 1, 'images': 1, 'subdirectories': 1},
 'totk_builds': {'videos': 1, 'images': 1, 'subdirectories': 0}}

### 2. Batch Processing Preparation

In [50]:
def prepare_batch_processing_list(media_dir, file_type='video'):
    """Prepare a list of files for batch processing with organized metadata"""
    
    if file_type == 'video':
        files = media_dir.all_video_files()
    elif file_type == 'image':
        files = media_dir.all_image_files()
    else:
        raise ValueError("file_type must be 'video' or 'image'")
    
    processing_list = []
    for file_obj in files:
        # Get relative path for organization
        rel_path = file_obj.path.relative_to(media_dir.path)
        
        # Determine category based on directory structure
        path_parts = rel_path.parts
        category = path_parts[0] if len(path_parts) > 1 else 'root'
        subcategory = path_parts[1] if len(path_parts) > 2 else None
        
        processing_list.append({
            'file_object': file_obj,
            'full_path': file_obj.path,
            'relative_path': rel_path,
            'category': category,
            'subcategory': subcategory,
            'filename': file_obj.path.name,
            'extension': file_obj.path.suffix.lower()
        })
    
    return processing_list

In [51]:
# Prepare video processing list
video_batch = prepare_batch_processing_list(media_dir, 'video')

In [52]:
# Show batch structure (first few items)
[(item['category'], item['subcategory'], item['filename']) for item in video_batch[:3]]

[('battles', 'totk_lynels', 'how_to_battle.mp4'),
 ('totk_builds', None, 'op_builds.mp4')]

In [53]:
# Group by category for organized processing
by_category = {}
for item in video_batch:
    category = item['category']
    if category not in by_category:
        by_category[category] = []
    by_category[category].append(item)

In [54]:
# Show files grouped by category
{category: len(items) for category, items in by_category.items()}

{'battles': 1, 'totk_builds': 1}

### 3. Change Detection and Backup Verification

In [55]:
def create_backup_verification_report(source_dir, backup_dir):
    """Create a detailed report comparing source and backup directories"""
    
    removed, added = source_dir.file_diff(backup_dir)
    
    report = {
        'backup_complete': len(removed) == 0 and len(added) == 0,
        'missing_from_backup': removed,
        'extra_in_backup': added,
        'source_stats': {
            'total_files': len(source_dir.all_file_paths()),
            'videos': len(source_dir.all_video_files()),
            'images': len(source_dir.all_image_files())
        },
        'backup_stats': {
            'total_files': len(backup_dir.all_file_paths()),
            'videos': len(backup_dir.all_video_files()),
            'images': len(backup_dir.all_image_files())
        }
    }
    
    return report

In [56]:
# Compare original with modified directory (simulating backup check)
verification = create_backup_verification_report(media_dir, modified_media_dir)

In [57]:
# Show backup verification results
verification['backup_complete'], verification['source_stats'], verification['backup_stats']

(False,
 {'total_files': 4, 'videos': 2, 'images': 2},
 {'total_files': 4, 'videos': 2, 'images': 2})

In [58]:
# Show missing files (first few)
list(verification['missing_from_backup'])[:3]

[PosixPath('/tmp/tmprx75laht/totk_builds/op_builds_thumb.jpg'),
 PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle.mp4'),
 PosixPath('/tmp/tmprx75laht/battles/totk_lynels/how_to_battle_thumb.jpg')]

In [59]:
# Show extra files (first few)
list(verification['extra_in_backup'])[:3]

[PosixPath('/tmp/mediatools_modified_icxpolo4/new_dir/new_thumb.jpg'),
 PosixPath('/tmp/mediatools_modified_icxpolo4/battles/totk_lynels/how_to_battle.mp4'),
 PosixPath('/tmp/mediatools_modified_icxpolo4/new_vid.mp4')]

## Cleanup

Let's clean up our temporary directories.

In [61]:
# Clean up temporary directories
try:
    shutil.rmtree(temp_path)
    shutil.rmtree(modified_root)
    print("Temporary directories cleaned up successfully.")
except Exception as e:
    f"Cleanup warning: {e}"